# Dwarf Example 04: All 24 Omega-Ready Dwarfs

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

24 LITTLE THINGS galaxies are 'omega-ready': they have
per-ring rotation curves with n_points >= 5 and regular kinematics.
All 24 show negative outer gaps — consistent with the SPARC result.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {{filename}}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {{filename}}")
        else:
            print(f"  Already present: {{filename}}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('dwarf_irregular_corpus_v1.json') as f:
    corpus = json.load(f)

omega_ready = [g for g in corpus['galaxies']
               if g.get('omega_ready') and g.get('data') and len(g['data']) >= 2]
print(f"Omega-ready dwarfs: {len(omega_ready)}")

results = []
for g in omega_ready:
    d  = g['data']
    R  = [p['Rad']  for p in d]
    V  = [p.get('Vrot', 0) for p in d]
    R1, V1 = R[0],  V[0]
    R2, V2 = R[-1], V[-1]
    if R1>0 and R2>0 and V1>0 and V2>0:
        omega    = (V2/R2 - V1/R1) * (R1/R2)**1.5
        V_adj_R2 = V2 - R2 * omega
        outer_gap= V_adj_R2 - V2  # simplified outer gap
        results.append({'galaxy': g['galaxy'], 'omega': omega,
                        'outer_gap': outer_gap, 'vmax': max(V),
                        'n_points': len(d)})

gaps = [r['outer_gap'] for r in results]
print(f"\nAll outer gaps negative: {all(g < 0 for g in gaps)}")
print(f"Mean outer gap: {np.mean(gaps):.1f} km/s")

print(f"\n{'Galaxy':<18} {'omega':>8} {'outer_gap':>10} {'Vmax':>6} {'N':>4}")
print('-' * 52)
for r in sorted(results, key=lambda x: x['omega']):
    print(f"{r['galaxy']:<18} {r['omega']:>8.2f} {r['outer_gap']:>10.2f} "
          f"{r['vmax']:>6.1f} {r['n_points']:>4}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

omegas = [r['omega'] for r in results]
names  = [r['galaxy'] for r in results]

axes[0].barh(range(len(omegas)),
             sorted(omegas),
             color=['#2ca02c' if o > 0 else '#d62728' for o in sorted(omegas)],
             alpha=0.8)
axes[0].axvline(7.06, color='blue', ls=':', lw=1.5, label='SPARC mean=7.06')
axes[0].axvline(9.94, color='red',  ls='--', lw=1.5, label='Dwarf median=9.94')
axes[0].set_xlabel(r'$\omega$ (rad/Gyr)', fontsize=11)
axes[0].set_title('Omega — All 24 Omega-Ready Dwarfs', fontsize=10)
axes[0].legend(fontsize=8)

axes[1].scatter([r['vmax'] for r in results], omegas,
                s=50, color='#2ca02c', alpha=0.8)
for r in results:
    axes[1].annotate(r['galaxy'][:6], (r['vmax'], r['omega']),
                     textcoords='offset points', xytext=(3,2), fontsize=6)
axes[1].set_xlabel(r'$V_{\rm max}$ (km/s)', fontsize=11)
axes[1].set_ylabel(r'$\omega$ (rad/Gyr)', fontsize=11)
axes[1].set_title('Omega vs Vmax', fontsize=10)

plt.suptitle('24 Omega-Ready Dwarfs — EPS Research Corpus v1.0\n'
             'Flynn (2026) | DOI: 10.5281/zenodo.20320362', fontsize=10)
plt.tight_layout()
plt.savefig('dw04_omega_ready.png', dpi=150, bbox_inches='tight')
plt.show()